In [4]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .appName("03_Bronze_to_Silver")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.ui.enabled", "false")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark OK :", spark.version)


Spark OK : 3.5.1


In [5]:
BRONZE = "C:/Users/smagu/retail-data-platform/data/bronze"
SILVER = "C:/Users/smagu/retail-data-platform/data/silver"

clients_pd      = pd.read_csv(BRONZE + "/clients.csv")
employes_pd     = pd.read_csv(BRONZE + "/employes.csv")
fournisseurs_pd = pd.read_csv(BRONZE + "/fournisseurs.csv")
produits_pd     = pd.read_csv(BRONZE + "/produits.csv")
ventes_pd       = pd.read_csv(BRONZE + "/ventes.csv")

print("Tables Bronze chargees :")
for nom, df in [("clients", clients_pd), ("employes", employes_pd),
                ("fournisseurs", fournisseurs_pd), ("produits", produits_pd),
                ("ventes", ventes_pd)]:
    print("  " + nom + " : " + str(len(df)) + " lignes")


Tables Bronze chargees :
  clients : 100 lignes
  employes : 100 lignes
  fournisseurs : 100 lignes
  produits : 100 lignes
  ventes : 100 lignes


In [6]:
# Nettoyage : types et colonnes derivees
clients_pd["ClientID"]           = clients_pd["ClientID"].astype(int)
employes_pd["EmployeID"]         = employes_pd["EmployeID"].astype(int)
fournisseurs_pd["FournisseurID"] = fournisseurs_pd["FournisseurID"].astype(int)
produits_pd["PrixUnitaire"]      = produits_pd["PrixUnitaire"].astype(float)

ventes_pd["DateVente"]      = pd.to_datetime(ventes_pd["DateVente"])
ventes_pd["Annee"]          = ventes_pd["DateVente"].dt.year
ventes_pd["Mois"]           = ventes_pd["DateVente"].dt.month
ventes_pd["MontantTotal"]   = ventes_pd["MontantTotal"].astype(float)
ventes_pd["QuantiteVendue"] = ventes_pd["QuantiteVendue"].astype(int)

print("Nettoyage de base OK")
print(ventes_pd.dtypes)


Nettoyage de base OK
VenteID                    int64
DateVente         datetime64[us]
ClientID                   int64
ProduitID                  int64
EmployeID                  int64
QuantiteVendue             int64
MontantTotal             float64
Annee                      int32
Mois                       int32
dtype: object


In [7]:
# Filtrage des valeurs aberrantes — methode IQR sur QuantiteVendue
Q1  = ventes_pd["QuantiteVendue"].quantile(0.25)
Q3  = ventes_pd["QuantiteVendue"].quantile(0.75)
IQR = Q3 - Q1
seuil_haut = Q3 + 1.5 * IQR

outliers     = ventes_pd[ventes_pd["QuantiteVendue"] > seuil_haut]
ventes_clean = ventes_pd[ventes_pd["QuantiteVendue"] <= seuil_haut].copy()

print("=== DETECTION OUTLIERS (IQR sur QuantiteVendue) ===")
print("Q1=" + str(int(Q1)) + "  |  Q3=" + str(int(Q3)) + "  |  IQR=" + str(int(IQR)) + "  |  Seuil haut=" + str(int(seuil_haut)))
print("Lignes avant  : " + str(len(ventes_pd)))
print("Outliers      : " + str(len(outliers)))
print("Lignes apres  : " + str(len(ventes_clean)))
print()
print("Lignes supprimees :")
print(outliers[["VenteID", "ProduitID", "QuantiteVendue", "MontantTotal"]].to_string(index=False))


=== DETECTION OUTLIERS (IQR sur QuantiteVendue) ===
Q1=1541  |  Q3=9249  |  IQR=7707  |  Seuil haut=20809
Lignes avant  : 100
Outliers      : 14
Lignes apres  : 86

Lignes supprimees :
 VenteID  ProduitID  QuantiteVendue  MontantTotal
      14         87           23099    23075901.0
      19         70           23799    19015401.0
      21         18           34999    24464301.0
      22         71           22199     1886915.0
      33         15           48499     6256371.0
      34         54           27499     3547371.0
      37         24           22199    22176801.0
      46         71           45499     3867415.0
      63         88           45999    45953001.0
      66         62           43999    13155701.0
      71         70           44499    35554701.0
      76          6           30099     8999601.0
      81         51           21299    14888001.0
      86          4           23099     6906601.0


In [8]:
print("=== STATS MontantTotal AVANT filtrage ===")
print(ventes_pd["MontantTotal"].describe().apply(lambda x: "{:,.0f}".format(x)))
print()
print("=== STATS MontantTotal APRES filtrage ===")
print(ventes_clean["MontantTotal"].describe().apply(lambda x: "{:,.0f}".format(x)))


=== STATS MontantTotal AVANT filtrage ===
count           100
mean      3,900,848
std       7,492,441
min          17,940
25%         386,970
50%       1,280,580
75%       2,841,676
max      45,953,001
Name: MontantTotal, dtype: str

=== STATS MontantTotal APRES filtrage ===
count            86
mean      1,864,380
std       3,118,050
min          17,940
25%         330,406
50%       1,109,851
75%       2,032,975
max      19,479,501
Name: MontantTotal, dtype: str


In [9]:
# Ecriture Silver en Parquet
tables_silver = {
    "clients":      clients_pd,
    "employes":     employes_pd,
    "fournisseurs": fournisseurs_pd,
    "produits":     produits_pd,
    "ventes":       ventes_clean,
}

for nom, df in tables_silver.items():
    path = SILVER + "/" + nom
    os.makedirs(path, exist_ok=True)
    df.to_parquet(path + "/" + nom + ".parquet", index=False)
    print("Ecrit : " + path + "/" + nom + ".parquet  (" + str(len(df)) + " lignes)")

print("")
print("Toutes les tables Silver sont enregistrees !")


Ecrit : C:/Users/smagu/retail-data-platform/data/silver/clients/clients.parquet  (100 lignes)
Ecrit : C:/Users/smagu/retail-data-platform/data/silver/employes/employes.parquet  (100 lignes)
Ecrit : C:/Users/smagu/retail-data-platform/data/silver/fournisseurs/fournisseurs.parquet  (100 lignes)
Ecrit : C:/Users/smagu/retail-data-platform/data/silver/produits/produits.parquet  (100 lignes)
Ecrit : C:/Users/smagu/retail-data-platform/data/silver/ventes/ventes.parquet  (86 lignes)

Toutes les tables Silver sont enregistrees !


In [10]:
# Verification finale
print("=== VERIFICATION SILVER ===")
for nom in tables_silver.keys():
    df_check = pd.read_parquet(SILVER + "/" + nom + "/" + nom + ".parquet")
    print(nom.upper() + " — " + str(len(df_check)) + " lignes | colonnes : " + str(list(df_check.columns)))

print("")
print("Notebook 03 termine !")


=== VERIFICATION SILVER ===
CLIENTS — 100 lignes | colonnes : ['ClientID', 'Nom', 'Prenom', 'Adresse', 'Email', 'NumeroTelephone']
EMPLOYES — 100 lignes | colonnes : ['EmployeID', 'Nom', 'Prenom', 'Fonction', 'Email', 'NuméroTelephone']
FOURNISSEURS — 100 lignes | colonnes : ['FournisseurID', 'NomFournisseur', 'Adresse', 'Email', 'NumeroTelephone']
PRODUITS — 100 lignes | colonnes : ['ProduitID', 'NomProduit', 'Description', 'PrixUnitaire', 'FournisseurID']
VENTES — 86 lignes | colonnes : ['VenteID', 'DateVente', 'ClientID', 'ProduitID', 'EmployeID', 'QuantiteVendue', 'MontantTotal', 'Annee', 'Mois']

Notebook 03 termine !
